# Fase 2 - Obtencion, limpieza y transformacion de datos
Proyecto: Proyecto ABP - Ciencia de Datos Reproducible

Notebook ejecutable para el pipeline inicial de preprocesamiento.

## 1. Objetivo F2
Implementar un flujo reproducible para cargar, perfilar, limpiar, transformar y validar el dataset del proyecto.

In [1]:
import sys
from pathlib import Path
#libreria para procesamiento de datos
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parents[1]
sys.path.append(str(ROOT / 'F2' / 'src'))
from preprocessing import (  # noqa: E402
    load_dataset, profile_dataset, clean_dataset,
    transform_dataset, validate_dataset, run_pipeline,
)
RAW = ROOT / 'F2' / 'data' / 'raw' / 'dataset_base.csv'
OUT = ROOT / 'F2' / 'data' / 'processed' / 'dataset_procesado.csv'
print('Python:', sys.version.split()[0])
print('Pandas:', pd.__version__)
print('Dataset:', RAW)

Python: 3.12.0
Pandas: 3.0.3
Dataset: C:\Users\grave\abp_cienciadatos\F2\data\raw\dataset_base.csv


## 2. Obtencion y exploracion inicial
Se carga el dataset, se revisan dimensiones, tipos, nulos y duplicados. Este proceso se realiaza antes de la limpieza

In [2]:
df_raw = load_dataset(RAW)
display(df_raw.head())
print('Shape:', df_raw.shape)
display(profile_dataset(df_raw))

,id,segmento,region,antiguedad_meses,monto_operacion,dias_mora,canal,estado,score_riesgo
0,1,Empresa,Metropolitana,24.0,1500000.0,0.0,Web,Activo,0.12
1,2,Pyme,O'Higgins,12.0,850000.0,5.0,Sucursal,Activo,0.25
2,3,Empresa,Valparaiso,36.0,2300000.0,0.0,Ejecutivo,Activo,0.08
3,4,Microempresa,Metropolitana,6.0,NaN,15.0,Web,Revision,0.44
4,5,Pyme,O'Higgins,18.0,920000.0,NaN,Sucursal,Activo,0.22


Shape: (21, 9)


,tipo,nulos,nulos_pct,unicos
id,int64,0,0.00,20
segmento,str,0,0.00,4
region,str,0,0.00,5
antiguedad_meses,float64,1,4.76,19
monto_operacion,float64,1,4.76,19
dias_mora,float64,1,4.76,13
canal,str,0,0.00,5
estado,str,1,4.76,3
score_riesgo,float64,1,4.76,19


## 3. Limpieza
Criterios aplicados: eliminacion de duplicados, imputacion de numericas con mediana, normalizacion textual y relleno de categoricas no informadas.

In [3]:
df_clean = clean_dataset(df_raw)
display(df_clean.head())
display(profile_dataset(df_clean))
print(validate_dataset(df_clean))

,id,segmento,region,antiguedad_meses,monto_operacion,dias_mora,canal,estado,score_riesgo
0,1,Empresa,Metropolitana,24.0,1500000.0,0.0,Web,Activo,0.12
1,2,Pyme,O'Higgins,12.0,850000.0,5.0,Sucursal,Activo,0.25
2,3,Empresa,Valparaiso,36.0,2300000.0,0.0,Ejecutivo,Activo,0.08
3,4,Microempresa,Metropolitana,6.0,920000.0,15.0,Web,Revision,0.44
4,5,Pyme,O'Higgins,18.0,920000.0,4.0,Sucursal,Activo,0.22


,tipo,nulos,nulos_pct,unicos
id,int64,0,0.0,20
segmento,string,0,0.0,4
region,string,0,0.0,5
antiguedad_meses,float64,0,0.0,19
monto_operacion,float64,0,0.0,19
dias_mora,float64,0,0.0,13
canal,string,0,0.0,5
estado,string,0,0.0,4
score_riesgo,float64,0,0.0,19


{'filas': 20, 'columnas': 9, 'duplicados': 0, 'nulos_totales': 0, 'sin_nulos': True, 'sin_duplicados': True}


## 4. Transformacion
Se normalizan las variables numericas y se aplica One Hot Encoding a variables nominales.

In [4]:
df_transformed = transform_dataset(df_clean)
display(df_transformed.head())
print('Shape transformado:', df_transformed.shape)
print(validate_dataset(df_transformed))

,id,antiguedad_meses,monto_operacion,dias_mora,score_riesgo,antiguedad_meses_norm,monto_operacion_norm,dias_mora_norm,score_riesgo_norm,segmento_Empresa,...,region_metropolitana,canal_EJECUTIVO,canal_Ejecutivo,canal_Sucursal,canal_Web,canal_web,estado_Activo,estado_Moroso,estado_No informado,estado_Revision
0,1,24.0,1500000.0,0.0,0.12,0.357143,0.294118,0.000000,0.102564,1,...,0,0,0,0,1,0,1,0,0,0
1,2,12.0,850000.0,5.0,0.25,0.142857,0.141176,0.083333,0.269231,0,...,0,0,0,1,0,0,1,0,0,0
2,3,36.0,2300000.0,0.0,0.08,0.571429,0.482353,0.000000,0.051282,1,...,0,0,1,0,0,0,1,0,0,0
3,4,6.0,920000.0,15.0,0.44,0.035714,0.157647,0.250000,0.512821,0,...,0,0,0,0,1,0,0,0,0,1
4,5,18.0,920000.0,4.0,0.22,0.250000,0.157647,0.066667,0.230769,0,...,0,0,0,1,0,0,1,0,0,0


Shape transformado: (20, 27)
{'filas': 20, 'columnas': 27, 'duplicados': 0, 'nulos_totales': 0, 'sin_nulos': True, 'sin_duplicados': True}


## 5. Pipeline completo y persistencia
Se ejecuta el pipeline completo, se guarda el dataset procesado y se verifica la salida.

In [5]:
clean, transformed, validation = run_pipeline(RAW, OUT)
print(validation)
assert validation['sin_nulos'], 'El dataset procesado mantiene valores nulos.'
assert validation['sin_duplicados'], 'El dataset procesado mantiene duplicados.'
assert OUT.exists(), 'No se genero el archivo procesado.'
print('Dataset procesado guardado en:', OUT)

{'filas': 20, 'columnas': 27, 'duplicados': 0, 'nulos_totales': 0, 'sin_nulos': True, 'sin_duplicados': True}
Dataset procesado guardado en: C:\Users\grave\abp_cienciadatos\F2\data\processed\dataset_procesado.csv


## 6. Justificación técnica

Las decisiones de preprocesamiento aplicadas responden a las características del dataset (calidad de aire para Santiago) y a los objetivos analíticos del proyecto, que apuntan a obtener un conjunto reproducible, sin sesgos introducidos en la limpieza y apto para modelos de Machine Learning.

### 6.1 Imputación con mediana
Se imputaron los valores faltantes de las variables numéricas con la **mediana** en lugar de la media porque las mediciones ambientales (concentraciones de contaminantes como PM2.5, PM10, O3, etc.) típicamente presentan **distribuciones asimétricas y valores extremos** asociados a episodios de alta contaminación. La media es sensible a estos outliers y desplazaría el valor imputado, distorsionando la distribución original. La mediana, al ser un estadístico robusto, conserva la tendencia central real de cada variable y no se ve arrastrada por picos puntuales, evitando introducir un sesgo artificial en datos que luego alimentarán el análisis y el modelado.

### 6.2 Normalización MinMax
Se aplicó **escalamiento MinMax** para llevar todas las variables numéricas a un rango común $[0, 1]$. Las variables del dataset operan en **escalas y unidades muy distintas** (concentraciones en $\mu\text{g/m}^3$, variables meteorológicas, etc.), por lo que, sin normalizar, los atributos de mayor magnitud dominarían las métricas de distancia y los pesos de los algoritmos. MinMax preserva la forma de la distribución y las relaciones relativas entre observaciones, además de ser apropiado cuando los datos no son estrictamente gaussianos. Esto garantiza una contribución equilibrada de cada variable en modelos sensibles a la escala (KNN, redes neuronales, métodos basados en gradiente) y facilita la convergencia durante el entrenamiento.

### 6.3 One Hot Encoding en variables nominales
Las variables **categóricas nominales** (por ejemplo, estación de monitoreo o categoría de calidad de aire) no poseen un orden intrínseco. Aplicar una codificación ordinal introduciría una **relación de magnitud inexistente** que los algoritmos interpretarían erróneamente como jerarquía o distancia entre categorías. El **One Hot Encoding** representa cada categoría como una variable binaria independiente, eliminando todo supuesto de orden y permitiendo que el modelo trate cada nivel de forma equitativa. Esto preserva la semántica real de las variables nominales y evita sesgos de codificación.

### 6.4 Relación con los objetivos analíticos
En conjunto, estas transformaciones aseguran un dataset **limpio, consistente, comparable y libre de sesgos** introducidos por la escala o por imputaciones poco robustas. Esto es coherente con el objetivo del proyecto de construir un pipeline **reproducible** para el análisis de calidad de aire en Santiago, dejando los datos en condiciones óptimas para las fases posteriores de exploración, modelado e interpretación de resultados.